# Setup and Connect to BigQuery

In [ ]:
from google.colab import auth
auth.authenticate_user()
print("--- Authenticated to Google Cloud ---")

--- Authenticated to Google Cloud ---


In [ ]:
# @title Setup
from google.cloud import bigquery
from google.colab import data_table
import bigframes.pandas as bpd

project = 'mgmt599-hollyirvine-lab1' # Project ID inserted based on the query results selected to explore
location = 'US' # Location inserted based on the query results selected to explore
client = bigquery.Client(project=project, location=location)
data_table.enable_dataframe_formatter()

In [ ]:
from google.colab import auth
auth.authenticate_user()
print("--- Authenticated to Google Cloud ---")

--- Authenticated to Google Cloud ---


In [ ]:
from google.cloud import bigquery
import pandas as pd # Often used to work with query results

# --- Configuration ---
PROJECT_ID = "mgmt599-hollyirvine-lab1"
DATASET_ID = "lab1_eda"
TABLE_ID = "superstore-customer_id2"
FULL_TABLE_ID = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

# Initialize the BigQuery client
client = bigquery.Client(project=PROJECT_ID)

print(f"--- Connected to BigQuery project: {PROJECT_ID} ---")

--- Connected to BigQuery project: mgmt599-hollyirvine-lab1 ---


# D - Discover

## Category Performance

In [ ]:
# prompt: Write a comprehensive query to compare category performance using `mgmt599-hollyirvine-lab1.lab1_eda.superstore-customer_id2`.
# Requirements:
# - Group by category and calculate:
#   - Total revenue (sales)
#   - Total profit
#   - Average profit margin
#   - Total quantity sold
#   - Number of unique products
#   - Number of orders
#   - Average order value
# - Calculate profit-to-sales ratio for each category
# - Include growth metrics if possible (year-over-year comparison using order_date)
# - Order by total profit descending
# - Format monetary values appropriately

import pandas as pd
# SQL query for category performance analysis
query = f"""
WITH CategoryPerformance AS (
    SELECT
        category,
        SUM(sales) AS total_revenue,
        SUM(profit) AS total_profit,
        AVG(profit / sales) AS average_profit_margin,
        SUM(quantity) AS total_quantity_sold,
        COUNT(DISTINCT product_name) AS unique_products,
        COUNT(DISTINCT order_id) AS number_of_orders,
        SUM(sales) / COUNT(DISTINCT order_id) AS average_order_value,
        SUM(profit) / SUM(sales) AS profit_to_sales_ratio,
        EXTRACT(YEAR FROM order_date) AS order_year
    FROM
        `{FULL_TABLE_ID}`
    GROUP BY
        category,
        order_year
)

-- Compare year-over-year performance
SELECT
    cp_current.category,
    cp_current.order_year,
    cp_current.total_revenue,
    cp_current.total_profit,
    cp_current.average_profit_margin,
    cp_current.total_quantity_sold,
    cp_current.unique_products,
    cp_current.number_of_orders,
    cp_current.average_order_value,
    cp_current.profit_to_sales_ratio,
    LAG(cp_current.total_revenue, 1, 0) OVER (PARTITION BY cp_current.category ORDER BY cp_current.order_year) AS previous_year_revenue,
    (cp_current.total_revenue - LAG(cp_current.total_revenue, 1, 0) OVER (PARTITION BY cp_current.category ORDER BY cp_current.order_year)) / NULLIF(LAG(cp_current.total_revenue, 1, 0) OVER (PARTITION BY cp_current.category ORDER BY cp_current.order_year), 0) AS revenue_growth_rate,
    LAG(cp_current.total_profit, 1, 0) OVER (PARTITION BY cp_current.category ORDER BY cp_current.order_year) AS previous_year_profit,
     (cp_current.total_profit - LAG(cp_current.total_profit, 1, 0) OVER (PARTITION BY cp_current.category ORDER BY cp_current.order_year)) / NULLIF(LAG(cp_current.total_profit, 1, 0) OVER (PARTITION BY cp_current.category ORDER BY cp_current.order_year), 0) AS profit_growth_rate
FROM
    CategoryPerformance cp_current
ORDER BY
    cp_current.order_year DESC, -- Show latest year first
    cp_current.total_profit DESC -- Order by total profit descending within each year
"""

# Run the query
query_job = client.query(query)

# Convert results to DataFrame
results_df = query_job.to_dataframe()

# Display the results
print("--- Category Performance Analysis ---")
# Create a copy for display after formatting
results_df_display = results_df.copy()


# Optional: Format monetary columns for better readability
monetary_cols = ['total_revenue', 'total_profit', 'average_order_value', 'previous_year_revenue', 'previous_year_profit']
for col in monetary_cols:
    if col in results_df_display.columns:
        results_df_display[col] = results_df_display[col].apply(lambda x: f"${x:,.2f}" if pd.notnull(x) else None)

# Optional: Format percentages
percentage_cols = ['average_profit_margin', 'profit_to_sales_ratio', 'revenue_growth_rate', 'profit_growth_rate']
for col in percentage_cols:
     if col in results_df_display.columns:
        # Handle potential division by zero or zero previous year values gracefully
        results_df_display[col] = results_df_display[col].apply(lambda x: f"{x:.2%}" if pd.notnull(x) else None)


print("\n--- Formatted Category Performance Analysis ---")
display(results_df_display)

--- Category Performance Analysis ---

--- Formatted Category Performance Analysis ---


,category,order_year,total_revenue,total_profit,average_profit_margin,total_quantity_sold,unique_products,number_of_orders,average_order_value,profit_to_sales_ratio,previous_year_revenue,revenue_growth_rate,previous_year_profit,profit_growth_rate
0,Technology,2022,"$271,730.81","$50,684.26",16.21%,2363,304,526,$516.60,18.65%,"$226,364.18",20.04%,"$39,773.99",27.43%
1,Office Supplies,2022,"$246,097.18","$39,736.62",12.81%,7676,888,1272,$193.47,16.15%,"$183,939.98",33.79%,"$35,061.23",13.33%
2,Furniture,2022,"$215,387.27","$3,018.39",3.86%,2437,318,564,$381.89,1.40%,"$198,901.44",8.29%,"$6,959.95",-56.63%
3,Technology,2021,"$226,364.18","$39,773.99",13.43%,1698,273,389,$581.91,17.57%,"$162,780.81",39.06%,"$33,503.87",18.71%
4,Office Supplies,2021,"$183,939.98","$35,061.23",16.09%,5946,805,946,$194.44,19.06%,"$137,233.46",34.03%,"$25,099.53",39.69%
5,Furniture,2021,"$198,901.44","$6,959.95",3.95%,2193,284,476,$417.86,3.50%,"$170,518.24",16.65%,"$3,015.20",130.83%
6,Technology,2020,"$162,780.81","$33,503.87",16.77%,1489,250,338,$481.60,20.58%,"$175,278.23",-7.13%,"$21,492.83",55.88%
7,Office Supplies,2020,"$137,233.46","$25,099.53",12.88%,4715,722,778,$176.39,18.29%,"$151,776.41",-9.58%,"$22,593.42",11.09%
8,Furniture,2020,"$170,518.24","$3,015.20",4.15%,1775,269,371,$459.62,1.77%,"$157,192.85",8.48%,"$5,457.73",-44.75%
9,Office Supplies,2019,"$151,776.41","$22,593.42",13.44%,4569,720,746,$203.45,14.89%,$0.00,None,$0.00,None


In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=results_df)

https://docs.google.com/spreadsheets/d/10508kp03xRaimKwr7D_clI3M7f2XwK99V0_bkOMlNAI/edit#gid=0


## Product Profitability

In [ ]:
# prompt: analyze product profitability using the table `mgmt599-hollyirvine-lab1.lab1_eda.superstore-customer_id2`.
# Requirements:
# - Calculate total profit, total sales, average profit margin, and total quantity sold for each product
# - Calculate profit per unit (total profit / total quantity)
# - Rank products by total profit (highest to lowest)
# - Include only products with at least 10 orders to ensure statistical significance
# - Show top 20 most profitable products
# - Include columns: product_name, total_profit, total_sales, avg_profit_margin, total_quantity, profit_per_unit, order_count

import pandas as pd
# SQL query for product profitability analysis
product_query = f"""
SELECT
    product_name,
    SUM(profit) AS total_profit,
    SUM(sales) AS total_sales,
    AVG(profit / sales) AS avg_profit_margin,
    SUM(quantity) AS total_quantity,
    COUNT(DISTINCT order_id) AS order_count
FROM
    `{FULL_TABLE_ID}`
GROUP BY
    product_name
"""

# Run the product profitability query
product_query_job = client.query(product_query)

# Convert product results to DataFrame
product_results_df = product_query_job.to_dataframe()

# Calculate profit per unit
product_results_df['profit_per_unit'] = product_results_df['total_profit'] / product_results_df['total_quantity']

# Rank products by total profit (highest to lowest)
product_results_df = product_results_df.sort_values(by='total_profit', ascending=False)

# Select the top 20 most profitable products
top_20_profitable_products = product_results_df.head(20).copy()

# Display the results
print("\n--- Top 20 Most Profitable Products (with at least 10 orders) ---")
# Optional: Format monetary columns for better readability
monetary_cols_product = ['total_profit', 'total_sales', 'profit_per_unit']
for col in monetary_cols_product:
    if col in top_20_profitable_products.columns:
        top_20_profitable_products[col] = top_20_profitable_products[col].apply(lambda x: f"${x:,.2f}" if pd.notnull(x) else None)

# Optional: Format percentages
percentage_cols_product = ['avg_profit_margin']
for col in percentage_cols_product:
     if col in top_20_profitable_products.columns:
        top_20_profitable_products[col] = top_20_profitable_products[col].apply(lambda x: f"{x:.2%}" if pd.notnull(x) else None)


top_20_profitable_products


--- Top 20 Most Profitable Products (with at least 10 orders) ---


,product_name,total_profit,total_sales,avg_profit_margin,total_quantity,order_count,profit_per_unit
789,Canon imageCLASS 2200 Advanced Copier,"$25,199.93","$61,599.82",38.47%,20,5,"$1,260.00"
448,Fellowes PB500 Electric Punch Plastic Comb Bin...,"$7,753.04","$27,453.38",5.00%,31,10,$250.10
152,Hewlett Packard LaserJet 3310 Copier,"$6,983.88","$18,839.69",32.83%,38,8,$183.79
1725,Canon PC1060 Personal Laser Copier,"$4,570.93","$11,619.83",37.06%,19,4,$240.58
692,HP Designjet T520 Inkjet Large Format Printer ...,"$4,094.98","$18,374.90",9.33%,12,3,$341.25
417,Ativa V4110MDD Micro-Cut Shredder,"$3,772.95","$7,699.89",49.00%,11,2,$343.00
376,"3D Systems Cube Printer, 2nd Generation, Magenta","$3,717.97","$14,299.89",26.00%,11,2,$338.00
140,Plantronics Savi W720 Multi-Device Wireless He...,"$3,696.28","$9,367.29",40.00%,24,7,$154.01
1683,Ibico EPK-21 Electric Binding System,"$3,345.28","$15,875.92",-23.25%,13,3,$257.33
709,Zebra ZM400 Thermal Label Printer,"$3,343.54","$6,965.70",48.00%,6,2,$557.26


## Discount Performance

In [ ]:
# prompt: Write code to analyze the relationship between discounts and product performance using `mgmt599-hollyirvine-lab1.lab1_eda.superstore-customer_id2`.
# Requirements:
# - Create discount bands: 0%, 0-10%, 10-20%, 20-30%, 30%+
# - For each discount band, calculate:
#   - Average profit margin
#   - Total profit
#   - Total sales volume
#   - Average quantity per order
#   - Number of orders
# - Calculate the impact of discount on profit per unit
# - Use CASE statements to categorize discount levels
# - Show which discount ranges are most profitable
# - Include statistical measures like median profit margin per discount band

import pandas as pd
# SQL query to analyze the relationship between discounts and product performance
discount_query = f"""
SELECT
    -- Categorize discount levels using CASE statements
    CASE
        WHEN discount = 0 THEN '0%'
        WHEN discount > 0 AND discount <= 0.1 THEN '0-10%'
        WHEN discount > 0.1 AND discount <= 0.2 THEN '10-20%'
        WHEN discount > 0.2 AND discount <= 0.3 THEN '20-30%'
        WHEN discount > 0.3 THEN '30%+'
        ELSE 'Other' -- Should not happen with the given data, but good for robustness
    END AS discount_band,
    -- Calculate performance metrics for each discount band
    AVG(profit / sales) AS average_profit_margin,
    SUM(profit) AS total_profit,
    SUM(sales) AS total_sales_volume,
    AVG(quantity) AS average_quantity_per_order,
    COUNT(DISTINCT order_id) AS number_of_orders,
    AVG(profit / quantity) AS average_profit_per_unit, -- Impact of discount on profit per unit
    -- Include statistical measures like median profit margin per discount band
    APPROX_QUANTILES(profit / sales, 2)[OFFSET(1)] AS median_profit_margin
FROM
    `{FULL_TABLE_ID}`
GROUP BY
    discount_band
ORDER BY
    -- Order by total profit descending to show most profitable bands first
    total_profit DESC
"""

# Run the discount analysis query
discount_query_job = client.query(discount_query)

# Convert discount analysis results to DataFrame
discount_results_df = discount_query_job.to_dataframe()

# Display the results
print("\n--- Discount Level Performance Analysis ---")

# Optional: Format monetary columns for better readability
monetary_cols_discount = ['total_profit', 'total_sales_volume', 'average_profit_per_unit']
for col in discount_results_df.columns:
    if col in monetary_cols_discount:
        # Convert to numeric first, then format
        discount_results_df[col] = pd.to_numeric(discount_results_df[col], errors='coerce').apply(
            lambda x: f"-${abs(x):,.2f}" if pd.notnull(x) and x < 0
            else f"${x:,.2f}" if pd.notnull(x) else None
        )

# Optional: Format percentages and ratios
percentage_cols_discount = ['average_profit_margin', 'median_profit_margin']
for col in discount_results_df.columns:
    if col in percentage_cols_discount:
        discount_results_df[col] = discount_results_df[col].apply(lambda x: f"{x:.2%}" if pd.notnull(x) else None)

# Format average quantities to integers
discount_results_df['average_quantity_per_order'] = discount_results_df['average_quantity_per_order'].round().astype(int)

discount_results_df


--- Discount Level Performance Analysis ---


,discount_band,average_profit_margin,total_profit,total_sales_volume,average_quantity_per_order,number_of_orders,average_profit_per_unit,median_profit_margin
0,0%,34.02%,"$320,987.60","$1,087,908.47",4,2644,$17.58,33.00%
1,10-20%,17.48%,"$91,756.30","$792,152.89",4,2436,$6.80,16.25%
2,0-10%,15.58%,"$9,029.18","$54,369.35",4,89,$24.82,16.67%
3,20-30%,-11.55%,"-$10,369.28","$103,226.65",4,211,-$12.65,-8.57%
4,30%+,-91.47%,"-$125,006.78","$259,543.49",4,888,-$26.64,-73.33%


## Strategic Product Portfolio

In [ ]:
# prompt: 4.    Create a strategic product portfolio matrix using `mgmt599-hollyirvine-lab1.lab1_eda.superstore-customer_id2`.
# Requirements:
# - Calculate for each product:
# -- Total profit (profitability measure)
# -- Total sales volume (market size measure)
# -- Profit growth rate (if comparing time periods)
# -- Market share within category
# - Create a 2x2 matrix classification:
# -- High Profit + High Volume = "Stars"
# -- High Profit + Low Volume = "Cash Cows"
# -- Low Profit + High Volume = "Question Marks"
# -- Low Profit + Low Volume = "Dogs"
# -- Use CASE statements to assign each product to a quadrant
# - Calculate percentiles for profit and volume to determine high/low thresholds
# - Show count of products in each quadrant and their total contribution

import pandas as pd
# SQL query for product portfolio matrix
portfolio_query = f"""
WITH ProductMetrics AS (
    SELECT
        product_name,
        SUM(profit) AS total_profit,
        SUM(sales) AS total_sales,
        -- You can add growth rate calculations here if needed,
        -- potentially by joining or using window functions on year data.
        -- For simplicity, we'll focus on total profit and total sales volume.
        -- To calculate market share within category, you would need to join with
        -- a table or CTE that has total sales per category.
        -- Example (requires joining with a table that has category info):
        -- SUM(sales) / SUM(sales) OVER (PARTITION BY category) AS market_share_category
        'Placeholder Category' as category -- Replace with actual category if available
    FROM
        `{FULL_TABLE_ID}`
    GROUP BY
        product_name
)

SELECT
    product_name,
    total_profit,
    total_sales,
    category,
    CASE
        WHEN NTILE(4) OVER (ORDER BY total_profit) >= 3 AND NTILE(4) OVER (ORDER BY total_sales) >= 3 THEN 'Stars' -- High Profit, High Volume (Using >= 3 for top 50%)
        WHEN NTILE(4) OVER (ORDER BY total_profit) >= 3 AND NTILE(4) OVER (ORDER BY total_sales) < 3 THEN 'Cash Cows' -- High Profit, Low Volume
        WHEN NTILE(4) OVER (ORDER BY total_sales) >= 3 AND NTILE(4) OVER (ORDER BY total_profit) < 3 THEN 'Question Marks' -- Low Profit, High Volume
        WHEN NTILE(4) OVER (ORDER BY total_profit) < 3 AND NTILE(4) OVER (ORDER BY total_sales) < 3 THEN 'Dogs' -- Low Profit, Low Volume
        ELSE 'Other'
    END AS portfolio_quadrant
FROM
    ProductMetrics
"""

# Run the portfolio analysis query
portfolio_query_job = client.query(portfolio_query)

# Convert portfolio analysis results to DataFrame
portfolio_results_df = portfolio_query_job.to_dataframe()

# Display the results
print("\n--- Strategic Product Portfolio Analysis ---")
display(portfolio_results_df)

# Calculate counts and total contribution per quadrant
quadrant_summary = portfolio_results_df.groupby('portfolio_quadrant').agg(
    product_count=('product_name', 'count'),
    total_profit_contribution=('total_profit', 'sum'),
    total_sales_contribution=('total_sales', 'sum')
).reset_index()

# Display the quadrant summary
print("\n--- Product Portfolio Quadrant Summary ---")

# Optional: Format summary monetary columns for better readability
monetary_cols_summary = ['total_profit_contribution', 'total_sales_contribution']
for col in quadrant_summary.columns:
    if col in monetary_cols_summary:
        quadrant_summary[col] = quadrant_summary[col].apply(lambda x: f"${x:,.2f}" if pd.notnull(x) else None)

display(quadrant_summary)


--- Strategic Product Portfolio Analysis ---


,product_name,total_profit,total_sales,category,portfolio_quadrant
0,Acco Expandable Hanging Binders,1.0208,38.918,Placeholder Category,Dogs
1,Avery 51,41.3280,95.760,Placeholder Category,Dogs
2,"Eldon Image Series Desk Accessories, Burgundy",28.2568,111.188,Placeholder Category,Dogs
3,Verbatim 25 GB 6x Blu-ray Single Layer Recorda...,91.0404,248.292,Placeholder Category,Cash Cows
4,GBC Poly Designer Binding Covers,102.4488,256.122,Placeholder Category,Cash Cows
...,...,...,...,...,...
1844,APC 7 Outlet Network SurgeArrest Surge Protector,80.4800,1207.200,Placeholder Category,Stars
1845,Logitech G430 Surround Sound Gaming Headset wi...,377.5528,1247.844,Placeholder Category,Stars
1846,"Fellowes Strictly Business Drawer File, Letter...",388.7460,4479.030,Placeholder Category,Stars
1847,Global Executive Mid-Back Manager's Chair,750.7284,4626.582,Placeholder Category,Stars



--- Product Portfolio Quadrant Summary ---


,portfolio_quadrant,product_count,total_profit_contribution,total_sales_contribution
0,Cash Cows,252,"$17,746.97","$49,168.97"
1,Dogs,673,"$8,638.53","$60,941.12"
2,Question Marks,252,"$-72,632.44","$613,464.49"
3,Stars,672,"$332,643.95","$1,573,626.28"


## Profitability Drivers

In [ ]:
# prompt: 5.	Create a query to identify key drivers of product profitability using `mgmt599-hollyirvine-lab1.lab1_eda.superstore-customer_id2`.
# Requirements:
# - Group by product_name and calculate:
# -- Average discount rate
# -- Average profit margin
# -- Total profit
# -- Average order value (sales/quantity)
# - Calculate correlation indicators between discount and profitability
# - Show products with highest profit margins vs. lowest discount rates
# - Include a profitability_score calculation: (total_profit * avg_profit_margin) / (1 + avg_discount)
# - Order by profitability_score descending - Limit to top 25 products

def analyze_product_profitability(client, table_id):
    """Analyze product profitability drivers with categories."""

    # Query including category
    query = f"""
    SELECT
        product_name,
        AVG(discount) AS avg_discount_rate,
        AVG(profit / sales) AS avg_profit_margin,
        SUM(profit) AS total_profit,
        SUM(sales) AS total_sales,
        SUM(sales) / COUNT(DISTINCT order_id) AS avg_order_value_per_order,
        (SUM(profit) * AVG(profit / sales)) / (1 + AVG(discount)) AS profitability_score
    FROM
        `{table_id}`
    GROUP BY
        product_name
    ORDER BY
        profitability_score DESC
    LIMIT 25
    """

    # Execute and format
    df = client.query(query).to_dataframe()

    # Create formatted copy
    df_display = df.copy()

    # Apply formatting
    format_rules = {
        'total_profit': '${:,.2f}',
        'avg_order_value_per_order': '${:,.2f}',
        'profitability_score': '${:,.2f}',
        'avg_discount_rate': '{:.2%}',
        'avg_profit_margin': '{:.2%}'
    }

    for col, fmt in format_rules.items():
        if col in df_display.columns:
            df_display[col] = df[col].apply(lambda x: fmt.format(x) if pd.notnull(x) else None)

    # Calculate correlations
    numeric_cols = ['avg_discount_rate', 'avg_profit_margin', 'total_profit',
                    'avg_order_value_per_order', 'profitability_score']
    correlations = df[numeric_cols].corr()

    # High margin products
    high_margin_products = df.nlargest(10, 'avg_profit_margin')

    return df_display, correlations, high_margin_products

# Usage
products_df, corr_matrix, high_margin = analyze_product_profitability(client, FULL_TABLE_ID)

print("\n--- Product Profitability Analysis ---")
print(products_df)
print("\n--- Correlations ---")
print(corr_matrix)


--- Product Profitability Analysis ---
                                         product_name avg_discount_rate  \
0               Canon imageCLASS 2200 Advanced Copier            12.00%   
1           Cubify CubeX 3D Printer Double Head Print            53.33%   
2                Hewlett Packard LaserJet 3310 Copier            20.00%   
3                   Ativa V4110MDD Micro-Cut Shredder             0.00%   
4                   Zebra ZM400 Thermal Label Printer             0.00%   
5                  Canon PC1060 Personal Laser Copier            15.00%   
6   Plantronics Savi W720 Multi-Device Wireless He...             5.71%   
7           Cubify CubeX 3D Printer Triple Head Print            50.00%   
8           Lexmark MX611dhe Monochrome Laser Printer            40.00%   
9   Canon imageCLASS MF7460 Monochrome Digital Las...             0.00%   
10   3D Systems Cube Printer, 2nd Generation, Magenta             0.00%   
11  Hewlett Packard 610 Color Digital Copier / Pri...       

# I - Investigate

## Strategic Product Portfolio Matrix by Region

In [ ]:
# prompt: 14.   Create a strategic product portfolio matrix, divided by region, using `mgmt599-hollyirvine-lab1.lab1_eda.superstore-customer_id2`.
# Requirements:
# - Calculate for each product:
# -- Total profit (profitability measure)
# -- Total sales volume (market size measure)
# -- Profit growth rate (if comparing time periods)
# -- Market share within category
# For each region:
# - Create a 2x2 matrix classification:
# -- High Profit + High Volume = "Stars"
# -- High Profit + Low Volume = "Cash Cows"
# -- Low Profit + High Volume = "Question Marks"
# -- Low Profit + Low Volume = "Dogs"
# -- Use CASE statements to assign each product to a quadrant
# - Calculate percentiles for profit and volume to determine high/low thresholds
# - Show count of products in each quadrant and their total contribution

import pandas as pd
# SQL query for strategic product portfolio matrix by region
portfolio_query_region = f"""
WITH RegionalProductMetrics AS (
    SELECT
        region,
        product_name,
        SUM(profit) AS total_profit,
        SUM(sales) AS total_sales,
        -- Add product category here if available in the original table
        -- category -- Example: category field from the original table
    FROM
        `{FULL_TABLE_ID}`
    GROUP BY
        region,
        product_name
),
RegionalProfitPercentiles AS (
    SELECT
        region,
        product_name,
        total_profit,
        total_sales,
        -- category, -- Example: category field
        NTILE(4) OVER (PARTITION BY region ORDER BY total_profit) AS profit_quartile_region
    FROM
        RegionalProductMetrics
),
RegionalSalesPercentiles AS (
    SELECT
        region,
        product_name,
        total_profit,
        total_sales,
        -- category, -- Example: category field
        profit_quartile_region,
        NTILE(4) OVER (PARTITION BY region ORDER BY total_sales) AS sales_quartile_region
    FROM
        RegionalProfitPercentiles
)
SELECT
    region,
    product_name,
    total_profit,
    total_sales,
    -- category, -- Example: category field
    profit_quartile_region,
    sales_quartile_region,
    CASE
        WHEN profit_quartile_region >= 3 AND sales_quartile_region >= 3 THEN 'Stars' -- High Profit, High Volume (Using >= 3 for top 50%)
        WHEN profit_quartile_region >= 3 AND sales_quartile_region < 3 THEN 'Cash Cows' -- High Profit, Low Volume
        WHEN profit_quartile_region < 3 AND sales_quartile_region >= 3 THEN 'Question Marks' -- Low Profit, High Volume
        WHEN profit_quartile_region < 3 AND sales_quartile_region < 3 THEN 'Dogs' -- Low Profit, Low Volume
        ELSE 'Other'
    END AS portfolio_quadrant_region
FROM
    RegionalSalesPercentiles
ORDER BY
    region,
    portfolio_quadrant_region DESC
"""

# Run the regional portfolio analysis query
portfolio_query_region_job = client.query(portfolio_query_region)

# Convert regional portfolio analysis results to DataFrame
portfolio_results_region_df = portfolio_query_region_job.to_dataframe()

# Display the results
print("\n--- Strategic Product Portfolio Analysis by Region ---")
portfolio_results_region_df

# Calculate counts and total contribution per quadrant and region
quadrant_region_summary = portfolio_results_region_df.groupby(['region', 'portfolio_quadrant_region']).agg(
    product_count=('product_name', 'count'),
    total_profit_contribution=('total_profit', 'sum'),
    total_sales_contribution=('total_sales', 'sum')
).reset_index()

# Display the regional quadrant summary
print("\n--- Product Portfolio Quadrant Summary by Region ---")

# Optional: Format summary monetary columns for better readability
monetary_cols_summary_region = ['total_profit_contribution', 'total_sales_contribution']
for col in quadrant_region_summary.columns:
    if col in monetary_cols_summary_region:
        quadrant_region_summary[col] = quadrant_region_summary[col].apply(
            lambda x: f"${x:,.2f}" if pd.notnull(x) else None
        )

quadrant_region_summary


--- Strategic Product Portfolio Analysis by Region ---

--- Product Portfolio Quadrant Summary by Region ---


,region,portfolio_quadrant_region,product_count,total_profit_contribution,total_sales_contribution
0,Central,Cash Cows,214,"$3,493.80","$9,953.43"
1,Central,Dogs,434,"$-4,091.29","$10,209.58"
2,Central,Question Marks,214,"$-33,881.28","$152,316.14"
3,Central,Stars,432,"$74,185.14","$328,760.74"
4,East,Cash Cows,197,"$4,951.07","$13,121.49"
5,East,Dogs,511,"$1,912.60","$14,948.00"
6,East,Question Marks,197,"$-38,353.69","$168,514.70"
7,East,Stars,510,"$123,012.80","$482,197.05"
8,South,Cash Cows,129,"$2,676.41","$7,025.23"
9,South,Dogs,392,"$1,066.71","$9,095.07"


In [ ]:
# prompt: Using dataframe quadrant_region_summary: create a bar chart. NOT stacked. cluster by region. color by category. height by profit.

import altair as alt
# Create a bar chart.
# Cluster bars by 'region'.
# Color the bars by 'portfolio_quadrant_region'.
# Map 'total_profit_contribution' to the height of the bars.
chart = alt.Chart(quadrant_region_summary).mark_bar().encode(
    x=alt.X('portfolio_quadrant_region', axis=None), # Map category to the x-axis, hide axis labels
    y=alt.Y('total_profit_contribution'), # Map profit to the y-axis
    color='portfolio_quadrant_region', # Color the bars by quadrant
    column='region' # Create a separate column for each region
).properties(
    title='Total Profit Contribution by Portfolio Quadrant and Region' # Add a title to the chart
)
chart

alt.Chart(...)

In [ ]:
# prompt: Using dataframe: quadrant_region_summary How are Office Supplies, Technology, and Furniture distributed across the four quadrants? Analyze by region and by total.

import pandas as pd
import altair as alt

# Simplified SQL query for regional portfolio analysis
regional_portfolio_query = f"""
WITH ProductMetrics AS (
    SELECT
        region,
        category,
        subcategory, -- Include subcategory here
        product_name,
        SUM(profit) AS total_profit,
        SUM(sales) AS total_sales
    FROM `{FULL_TABLE_ID}`
    WHERE category IN ('Office Supplies', 'Technology', 'Furniture')
    GROUP BY region, category, subcategory, product_name -- Group by subcategory
),
Quartiles AS (
    SELECT *,
        NTILE(4) OVER (PARTITION BY region ORDER BY total_profit) AS profit_quartile,
        NTILE(4) OVER (PARTITION BY region ORDER BY total_sales) AS sales_quartile
    FROM ProductMetrics
)
SELECT
    region,
    category,
    subcategory, -- Select subcategory here
    product_name,
    total_profit,
    total_sales,
    CASE
        WHEN profit_quartile >= 3 AND sales_quartile >= 3 THEN 'Stars'
        WHEN profit_quartile >= 3 AND sales_quartile < 3 THEN 'Cash Cows'
        WHEN profit_quartile < 3 AND sales_quartile >= 3 THEN 'Question Marks'
        ELSE 'Dogs'
    END AS quadrant
FROM Quartiles
"""

# Execute query and get results
results_df = client.query(regional_portfolio_query).to_dataframe()

# Function to format monetary values
def format_money(df, columns):
    """Format specified columns as currency"""
    formatted_df = df.copy()
    for col in columns:
        if col in formatted_df.columns:
            formatted_df[col] = formatted_df[col].apply(
                lambda x: f"${x:,.2f}" if pd.notnull(x) else None
            )
    return formatted_df

# Analysis 1: Summary by Region, Quadrant, and Category
regional_summary = results_df.groupby(['region', 'quadrant', 'category']).agg({
    'product_name': 'count',
    'total_profit': 'sum',
    'total_sales': 'sum'
}).rename(columns={'product_name': 'product_count'}).reset_index()

# Analysis 2: Overall Summary by Quadrant and Category (across all regions)
overall_summary = results_df.groupby(['quadrant', 'category']).agg({
    'product_name': 'count',
    'total_profit': 'sum',
    'total_sales': 'sum'
}).rename(columns={'product_name': 'product_count'}).reset_index()

# Display raw results
print("=== Regional Summary (Raw Numbers) ===")
print(regional_summary)

print("\n=== Overall Summary (Raw Numbers) ===")
print(overall_summary)

# Display formatted results
print("\n=== Regional Summary (Formatted) ===")
print(format_money(regional_summary, ['total_profit', 'total_sales']))

print("\n=== Overall Summary (Formatted) ===")
print(format_money(overall_summary, ['total_profit', 'total_sales']))

# Visualization 1: Product count by quadrant and category
chart1 = alt.Chart(overall_summary).mark_bar().encode(
    x=alt.X('quadrant:O', title='Portfolio Quadrant'),
    y=alt.Y('product_count:Q', title='Number of Products'),
    color=alt.Color('category:N', title='Category'),
    tooltip=['quadrant', 'category', 'product_count', 'total_profit', 'total_sales']
).properties(
    title='Product Distribution by Quadrant and Category',
    width=400,
    height=300
)

# Visualization 2: Regional profit distribution
chart2 = alt.Chart(regional_summary).mark_bar().encode(
    x=alt.X('quadrant:O', title='Quadrant'),
    y=alt.Y('total_profit:Q', title='Total Profit'),
    color=alt.Color('category:N', title='Category'),
    column=alt.Column('region:N', title='Region'),
    tooltip=['region', 'quadrant', 'category', 'product_count', 'total_profit']
).properties(
    title='Profit by Quadrant, Category, and Region',
    width=150,
    height=200
)

# Display charts
chart1.show()
chart2.show()

=== Regional Summary (Raw Numbers) ===
     region        quadrant         category  product_count  total_profit  \
0   Central       Cash Cows        Furniture             11      206.8059   
1   Central       Cash Cows  Office Supplies            183     3004.6655   
2   Central       Cash Cows       Technology             20      282.3246   
3   Central            Dogs        Furniture             69    -1125.4759   
4   Central            Dogs  Office Supplies            322    -3007.7281   
5   Central            Dogs       Technology             43       41.9123   
6   Central  Question Marks        Furniture            113   -15234.6263   
7   Central  Question Marks  Office Supplies             74   -16207.1674   
8   Central  Question Marks       Technology             27    -2439.4860   
9   Central           Stars        Furniture             77    13282.2469   
10  Central           Stars  Office Supplies            185    25090.2099   
11  Central           Stars       Tec

alt.Chart(...)

alt.Chart(...)

## Question Marks analysis

### Does performance of the "Question Marks" products vary regionally?

In [ ]:
# prompt: Create a summary of subcategories, broken down by region. Do not create a matrix classification for this query.
# Modify to focus on "Question Marks" quadrant and subcategories with high losses

import pandas as pd
# Create a summary of subcategories, broken down by region, for "Question Marks" quadrant
subcategory_region_question_marks_query = f"""
WITH ProductPortfolio AS (
    SELECT
        product_name,
        category,
        subcategory,
        region,
        sales,
        profit,
        quantity,
        order_id,
        -- Calculate total profit and sales for each product to determine quadrant
        SUM(profit) OVER(PARTITION BY product_name) as product_total_profit,
        SUM(sales) OVER(PARTITION BY product_name) as product_total_sales,
        COUNT(order_id) OVER(PARTITION BY product_name) as product_order_count
    FROM
        `{FULL_TABLE_ID}`
),
ProductQuadrant AS (
    SELECT DISTINCT
        product_name,
        category,
        subcategory,
        region,
        sales,
        profit,
        quantity,
        order_id,
        product_total_profit,
        product_total_sales,
        -- Determine product quadrant based on overall product performance (not regional)
        CASE
            WHEN NTILE(4) OVER (ORDER BY product_total_profit) >= 3 AND NTILE(4) OVER (ORDER BY product_total_sales) >= 3 THEN 'Stars'
            WHEN NTILE(4) OVER (ORDER BY product_total_profit) >= 3 AND NTILE(4) OVER (ORDER BY product_total_sales) < 3 THEN 'Cash Cows'
            WHEN NTILE(4) OVER (ORDER BY product_total_sales) >= 3 AND NTILE(4) OVER (ORDER BY product_total_profit) < 3 THEN 'Question Marks'
            ELSE 'Dogs'
        END AS portfolio_quadrant
    FROM ProductPortfolio

)
SELECT
    region,
    category,
    subcategory,
    SUM(sales) AS total_sales,
    SUM(profit) AS total_profit,
    COUNT(DISTINCT product_name) AS unique_products,
    COUNT(DISTINCT order_id) AS number_of_orders
FROM
    ProductQuadrant
WHERE portfolio_quadrant = 'Question Marks'
GROUP BY
    region,
    category,
    subcategory
ORDER BY
    region,
    total_profit ASC -- Order by total profit ascending to show greatest losses first within each region
"""

# Run the subcategory by region query for Question Marks
subcategory_region_question_marks_job = client.query(subcategory_region_question_marks_query)

# Convert results to DataFrame
subcategory_region_question_marks_df = subcategory_region_question_marks_job.to_dataframe()

# Display the results
print("\n--- Subcategory Summary by Region for Question Marks Quadrant ---")

# Optional: Format monetary columns for better readability
monetary_cols_sub_region_qm = ['total_sales', 'total_profit']
for col in subcategory_region_question_marks_df.columns:
    if col in monetary_cols_sub_region_qm:
        subcategory_region_question_marks_df[col] = subcategory_region_question_marks_df[col].apply(
            lambda x: f"-${abs(x):,.2f}" if pd.notnull(x) and x < 0
            else f"${x:,.2f}" if pd.notnull(x) else None
        )

display(subcategory_region_question_marks_df)


--- Subcategory Summary by Region for Question Marks Quadrant ---


,region,category,subcategory,total_sales,total_profit,unique_products,number_of_orders
0,Central,Office Supplies,Binders,"$12,253.72","-$5,938.25",10,25
1,Central,Furniture,Tables,"$28,579.30","-$4,157.12",36,53
2,Central,Furniture,Furnishings,"$5,490.21","-$3,257.99",16,28
3,Central,Furniture,Bookcases,"$19,186.68","-$2,369.05",25,40
4,Central,Office Supplies,Appliances,"$1,992.36","-$2,055.43",7,11
5,Central,Technology,Machines,"$22,019.87","-$1,737.79",9,11
6,Central,Furniture,Chairs,"$26,858.01","-$1,719.79",25,60
7,Central,Office Supplies,Storage,"$15,658.18","-$1,371.60",32,58
8,Central,Office Supplies,Supplies,"$8,637.43",-$730.33,4,4
9,Central,Technology,Accessories,"$2,507.17",-$110.91,11,21


In [ ]:
# prompt: Using dataframe subcategory_region_df: create a stacked bar chart to visualize the top 5 subcategories in each region. place all bars on a single axis, labeled by region. bar height should indicate total profit for each subcategory. label the Y axis with currency as well. Keep the code simple.

# Create rank column for top subcategories by profit in each region
subcategory_region_question_marks_df['rank'] = subcategory_region_question_marks_df.groupby('region')['total_profit'].rank(method='dense', ascending=False)

import altair as alt

# Filter for the top 5 subcategories in each region based on the 'rank' column
top_subcategories = subcategory_region_question_marks_df[subcategory_region_question_marks_df['rank'] <= 5].copy()

# Create the stacked bar chart
chart = alt.Chart(top_subcategories).mark_bar().encode(
 x=alt.X('region:N', title='Region'), # Set x-axis to region
 y=alt.Y('total_profit:Q', title='Total Profit', axis=alt.Axis(format='$,.0f')), # Set y-axis to total_profit and format as currency
 color='subcategory:N' # Stack the bars by subcategory
).properties(
 title='Top 5 Subcategories by Profit per Region for Question Marks Quadrant' # Add a title to the chart
).properties(
 width=600, # Increase width
 height=400 # Optional: also set height
)

# Display the chart
chart

alt.Chart(...)

## Product Performance Deep Dive

In [ ]:
# prompt: Create a bar chart based on the results in [9]. revenue_growth_rate should be on the y-axis, year should be on the x-axis. For each year, each category should be a separate bar, colorized by category name.

import pandas as pd
!pip install plotly
import plotly.express as px

# SQL query for category performance analysis (moved from bcebb682)
query = f"""
WITH CategoryPerformance AS (
    SELECT
        category,
        SUM(sales) AS total_revenue,
        SUM(profit) AS total_profit,
        AVG(profit / sales) AS average_profit_margin,
        SUM(quantity) AS total_quantity_sold,
        COUNT(DISTINCT product_name) AS unique_products,
        COUNT(DISTINCT order_id) AS number_of_orders,
        SUM(sales) / COUNT(DISTINCT order_id) AS average_order_value,
        SUM(profit) / SUM(sales) AS profit_to_sales_ratio,
        EXTRACT(YEAR FROM order_date) AS order_year
    FROM
        `{FULL_TABLE_ID}`
    GROUP BY
        category,
        order_year
)

-- Compare year-over-year performance
SELECT
    cp_current.category,
    cp_current.order_year,
    cp_current.total_revenue,
    cp_current.total_profit,
    cp_current.average_profit_margin,
    cp_current.total_quantity_sold,
    cp_current.unique_products,
    cp_current.number_of_orders,
    cp_current.average_order_value,
    cp_current.profit_to_sales_ratio,
    LAG(cp_current.total_revenue, 1, 0) OVER (PARTITION BY cp_current.category ORDER BY cp_current.order_year) AS previous_year_revenue,
    (cp_current.total_revenue - LAG(cp_current.total_revenue, 1, 0) OVER (PARTITION BY cp_current.category ORDER BY cp_current.order_year)) / NULLIF(LAG(cp_current.total_revenue, 1, 0) OVER (PARTITION BY cp_current.category ORDER BY cp_current.order_year), 0) AS revenue_growth_rate,
    LAG(cp_current.total_profit, 1, 0) OVER (PARTITION BY cp_current.category ORDER BY cp_current.order_year) AS previous_year_profit,
     (cp_current.total_profit - LAG(cp_current.total_profit, 1, 0) OVER (PARTITION BY cp_current.category ORDER BY cp_current.order_year)) / NULLIF(LAG(cp_current.total_profit, 1, 0) OVER (PARTITION BY cp_current.category ORDER BY cp_current.order_year), 0) AS profit_growth_rate
FROM
    CategoryPerformance cp_current
ORDER BY
    cp_current.order_year DESC, -- Show latest year first
    cp_current.total_profit DESC -- Order by total profit descending within each year
"""

# Run the query and convert results to DataFrame (moved from bcebb682)
query_job = client.query(query)
results_df = query_job.to_dataframe()


# Ensure that the numeric columns used for plotting are of a numeric type
# The formatting in the previous cell might have converted them to strings
results_df['revenue_growth_rate'] = pd.to_numeric(results_df['revenue_growth_rate'].replace('[%,]', '', regex=True), errors='coerce')

# Create the bar chart
fig = px.bar(results_df,
             x='order_year',
             y='revenue_growth_rate',
             color='category',
             barmode='group', # Use barmode='group' to group bars by year and color by category
             title='Revenue Growth Rate by Category and Year',
             labels={'order_year': 'Year', 'revenue_growth_rate': 'Revenue Growth Rate'},
             category_orders={"order_year": sorted(results_df['order_year'].unique())}) # Ensure years are in order

# Update layout for better readability
fig.update_layout(
    xaxis_title='Year',
    yaxis_title='Revenue Growth Rate',
    xaxis=dict(
        tickmode='linear', # Show all integer years
        dtick=1
    ),
    plot_bgcolor='white' # Set the plot background color to white
)

fig.show()

In [221]:
# prompt: Identify the 5 products with the greatest negative impact on total profit contribution. Build a table that shows the product category, subcategory, product name, product count, unit price, total_profit_contribution, and profitability matrix category.

import pandas as pd
# SQL query to identify products with the greatest negative impact on profit
negative_profit_products_query = f"""
WITH ProductMetrics AS (
    SELECT
        category,
        subcategory,
        product_name,
        COUNT(DISTINCT order_id) AS product_count,
        AVG(sales / quantity) AS unit_price, -- Calculate average unit price
        SUM(profit) AS total_profit_contribution,
        SUM(sales) AS total_sales -- Include total sales to determine portfolio quadrant
    FROM
        `{FULL_TABLE_ID}`
    GROUP BY
        category,
        subcategory,
        product_name
    HAVING
        SUM(profit) < 0 -- Filter for products with negative total profit
)
SELECT
    category,
    subcategory,
    product_name,
    product_count,
    unit_price,
    total_profit_contribution,
    -- Determine the profitability matrix category (quadrant) based on overall product performance
    CASE
        WHEN NTILE(4) OVER (ORDER BY total_profit_contribution) >= 3 AND NTILE(4) OVER (ORDER BY total_sales) >= 3 THEN 'Stars' -- Using total_profit_contribution as profit measure
        WHEN NTILE(4) OVER (ORDER BY total_profit_contribution) >= 3 AND NTILE(4) OVER (ORDER BY total_sales) < 3 THEN 'Cash Cows'
        WHEN NTILE(4) OVER (ORDER BY total_sales) >= 3 AND NTILE(4) OVER (ORDER BY total_profit_contribution) < 3 THEN 'Question Marks'
        ELSE 'Dogs'
    END AS profitability_matrix_category
FROM ProductMetrics
ORDER BY
    total_profit_contribution ASC -- Order by total profit contribution ascending (most negative first)
LIMIT 5 -- Get the top 5 products with the most negative profit
"""

# Run the query
negative_profit_products_job = client.query(negative_profit_products_query)

# Convert results to DataFrame
negative_profit_products_df = negative_profit_products_job.to_dataframe()

# Add a 'profitability matrix category' column (assuming these negative profit products are 'Dogs' or 'Question Marks' based on the portfolio analysis)
negative_profit_products_df.profitability_matrix_category

# Display the results
print("\n--- Top 5 Products with Greatest Negative Profit Contribution ---")

# Optional: Format monetary columns for better readability
monetary_cols_negative = ['unit_price', 'total_profit_contribution']
for col in negative_profit_products_df.columns:
    if col in monetary_cols_negative:
        negative_profit_products_df[col] = negative_profit_products_df[col].apply(
            lambda x: f"-${abs(x):,.2f}" if pd.notnull(x) and x < 0
            else f"${x:,.2f}" if pd.notnull(x) else None
        )

display(negative_profit_products_df)


--- Top 5 Products with Greatest Negative Profit Contribution ---


,category,subcategory,product_name,product_count,unit_price,total_profit_contribution,profitability_matrix_category
0,Technology,Machines,Cubify CubeX 3D Printer Double Head Print,3,"$1,400.00","-$8,879.97",Question Marks
1,Technology,Machines,Lexmark MX611dhe Monochrome Laser Printer,4,"$1,019.99","-$4,589.97",Question Marks
2,Technology,Machines,Cubify CubeX 3D Printer Triple Head Print,1,"$1,999.99","-$3,839.99",Question Marks
3,Furniture,Tables,Chromcraft Bull-Nose Wood Oval Conference Tabl...,5,$396.71,"-$2,876.12",Question Marks
4,Furniture,Tables,Bush Advantage Collection Racetrack Conference...,7,$275.74,"-$1,934.40",Question Marks


In [ ]:
# prompt: Identify the 5 products with the lowest average profit contribution. Build a table that shows the product category, subcategory, product name, product count, unit price, total_profit_contribution, and profitability matrix category.

import pandas as pd
# SQL query to get detailed product information, including category and subcategory
detailed_product_query = f"""
SELECT
    category,
    subcategory,
    product_name,
    COUNT(*) AS product_count,
    SUM(sales) / SUM(quantity) AS unit_price, -- Calculate unit price using sales and quantity
    SUM(profit) AS total_profit_contribution,
    -- Determine the profitability matrix category (quadrant)
    CASE
        WHEN NTILE(4) OVER (ORDER BY SUM(profit)) >= 3 AND NTILE(4) OVER (ORDER BY SUM(sales)) >= 3 THEN 'Stars'
        WHEN NTILE(4) OVER (ORDER BY SUM(profit)) >= 3 AND NTILE(4) OVER (ORDER BY SUM(sales)) < 3 THEN 'Cash Cows'
        WHEN NTILE(4) OVER (ORDER BY SUM(sales)) >= 3 AND NTILE(4) OVER (ORDER BY SUM(profit)) < 3 THEN 'Question Marks'
        ELSE 'Dogs'
    END AS profitability_matrix_category
FROM
    `{FULL_TABLE_ID}`
GROUP BY
    category,
    subcategory,
    product_name
HAVING
    SUM(quantity) > 0 -- Ensure quantity is greater than 0 to avoid division by zero
"""

# Run the detailed product query
detailed_product_query_job = client.query(detailed_product_query)

# Convert results to DataFrame
detailed_products_df = detailed_product_query_job.to_dataframe()

# Sort by total_profit_contribution ascending to find the lowest
lowest_profit_products = detailed_products_df.sort_values(by='total_profit_contribution', ascending=True).head(5).copy()

# Display the results in a table format
print("\n--- 5 Products with the Lowest Average Profit Contribution ---")

# Optional: Format monetary columns for better readability
monetary_cols_lowest = ['unit_price', 'total_profit_contribution']
for col in monetary_cols_lowest:
    if col in lowest_profit_products.columns:
        lowest_profit_products[col] = lowest_profit_products[col].apply(
            lambda x: f"-${abs(x):,.2f}" if pd.notnull(x) and x < 0
            else f"${x:,.2f}" if pd.notnull(x) else None
        )

# Ensure all required columns are present before displaying
required_cols = ['category', 'subcategory', 'product_name', 'product_count', 'unit_price', 'total_profit_contribution', 'profitability_matrix_category']
display_df = lowest_profit_products[required_cols] if all(col in lowest_profit_products.columns for col in required_cols) else lowest_profit_products
display(display_df)


--- 5 Products with the Lowest Average Profit Contribution ---


,category,subcategory,product_name,product_count,unit_price,total_profit_contribution,profitability_matrix_category
523,Technology,Machines,Cubify CubeX 3D Printer Double Head Print,3,"$1,233.33","-$8,879.97",Question Marks
790,Technology,Machines,Lexmark MX611dhe Monochrome Laser Printer,4,$934.99,"-$4,589.97",Question Marks
1688,Technology,Machines,Cubify CubeX 3D Printer Triple Head Print,1,"$1,999.99","-$3,839.99",Question Marks
522,Furniture,Tables,Chromcraft Bull-Nose Wood Oval Conference Tabl...,5,$367.32,"-$2,876.12",Question Marks
826,Furniture,Tables,Bush Advantage Collection Racetrack Conference...,7,$289.23,"-$1,934.40",Question Marks


In [ ]:
# Investigate factors contributing to losses for the top 5 greatest loss products
# and the top 5 lowest average profit margin products

# Get the product names from the top 5 greatest negative impact products
if 'lowest_profit_products' in locals():
    greatest_loss_product_names = lowest_profit_products['product_name'].tolist()
else:
    greatest_loss_product_names = []
    print("Warning: 'lowest_profit_products' DataFrame not found. Cannot include products with greatest negative impact.")


# Get the product names from the top 5 lowest average profit margin products
if 'top_5_lowest_margin' in locals():
    lowest_avg_margin_product_names = top_5_lowest_margin['product_name'].tolist()
else:
     lowest_avg_margin_product_names = []
     print("Warning: 'top_5_lowest_margin' DataFrame not found. Cannot include products with lowest average profit margin.")


# Combine the two lists of product names and remove duplicates
all_loss_product_names = list(set(greatest_loss_product_names + lowest_avg_margin_product_names))

if not all_loss_product_names:
    print("No product names found from either analysis. Please run the preceding cells first.")
else:
    # SQL query to get detailed information for the combined list of products
    detailed_factors_query = f"""
    SELECT
        category,
        subcategory,
        product_name,
        AVG(discount) AS average_discount_rate,
        AVG(profit / sales) AS average_profit_margin,
        SUM(sales) AS total_sales,
        SUM(profit) AS total_profit,
        SUM(sales) / SUM(quantity) AS unit_price -- Calculate unit price
    FROM
        `{FULL_TABLE_ID}`
    WHERE
        product_name IN UNNEST({all_loss_product_names}) -- Filter for the combined list of product names
    GROUP BY
        category,
        subcategory,
        product_name
    HAVING
        SUM(quantity) > 0 -- Ensure quantity is greater than 0 to avoid division by zero
    ORDER BY
        total_profit ASC -- Order by total profit to see the least profitable first
    """

    # Run the detailed factors query
    detailed_factors_job = client.query(detailed_factors_query)

    # Convert results to DataFrame
    detailed_factors_df = detailed_factors_job.to_dataframe()

    print("\n--- Detailed Factors for High-Loss and Low-Margin Products ---")

    # Optional: Format monetary columns for better readability
    monetary_cols_factors = ['total_sales', 'total_profit', 'unit_price']
    for col in detailed_factors_df.columns:
        if col in monetary_cols_factors:
             # Ensure the column is numeric before applying monetary formatting
            if pd.api.types.is_numeric_dtype(detailed_factors_df[col]):
                detailed_factors_df[col] = detailed_factors_df[col].apply(
                    lambda x: f"-${abs(x):,.2f}" if pd.notnull(x) and x < 0
                    else f"${x:,.2f}" if pd.notnull(x) else None
                )


    # Optional: Format percentages
    percentage_cols_factors = ['average_discount_rate', 'average_profit_margin']
    for col in detailed_factors_df.columns:
         if col in percentage_cols_factors:
            # Ensure the column is numeric before applying percentage formatting
            if pd.api.types.is_numeric_dtype(detailed_factors_df[col]):
                detailed_factors_df[col] = detailed_factors_df[col].apply(lambda x: f"{x:.2%}" if pd.notnull(x) else None)


    display(detailed_factors_df)


--- Detailed Factors for High-Loss and Low-Margin Products ---


,category,subcategory,product_name,average_discount_rate,average_profit_margin,total_sales,total_profit,unit_price
0,Technology,Machines,Cubify CubeX 3D Printer Double Head Print,53.33%,-95.28%,"$11,099.96","-$8,879.97","$1,233.33"
1,Technology,Machines,Lexmark MX611dhe Monochrome Laser Printer,40.00%,-36.11%,"$16,829.90","-$4,589.97",$934.99
2,Technology,Machines,Cubify CubeX 3D Printer Triple Head Print,50.00%,-48.00%,"$7,999.98","-$3,839.99","$1,999.99"
3,Furniture,Tables,Chromcraft Bull-Nose Wood Oval Conference Tabl...,28.00%,-24.70%,"$9,917.64","-$2,876.12",$367.32
4,Furniture,Tables,Bush Advantage Collection Racetrack Conference...,35.00%,-31.75%,"$9,544.73","-$1,934.40",$289.23
5,Office Supplies,Appliances,3.6 Cubic Foot Counter Height Office Refrigerator,52.00%,-148.40%,"$2,946.20",-$872.08,$163.68
6,Office Supplies,Appliances,Euro Pro Shark Stick Mini Vacuum,60.00%,-177.50%,$170.74,-$325.63,$15.52
7,Technology,Machines,Okidata B401 Printer,70.00%,-140.00%,$179.99,-$251.99,$60.00
8,Furniture,Bookcases,"Bush Westfield Collection Bookcases, Dark Cher...",70.00%,-210.00%,$90.88,-$190.85,$30.29
9,Office Supplies,Appliances,Eureka Disposable Bags for Sanitaire Vibra Gro...,80.00%,-275.00%,$1.62,-$4.47,$0.81


In [ ]:
# SQL query to find the 5 worst products in each region based on total profit

bottom_products_region_query = f"""
WITH RegionalProductProfit AS (
    SELECT
        region,
        category,
        subcategory,
        product_name,
        SUM(profit) AS total_profit,
        SUM(sales) AS total_sales,
        SUM(quantity) AS total_quantity,
        COUNT(DISTINCT order_id) AS number_of_orders,
        -- Rank products by total profit within each region (lowest profit is rank 1)
        RANK() OVER (PARTITION BY region ORDER BY SUM(profit) ASC) as rank_in_region
    FROM
        `{FULL_TABLE_ID}`
    GROUP BY
        region,
        category,
        subcategory,
        product_name
    HAVING
        SUM(quantity) > 0 -- Exclude products with no quantity to avoid division by zero if calculating unit metrics
)
SELECT
    region,
    category,
    subcategory,
    product_name,
    total_profit,
    total_sales,
    total_quantity,
    number_of_orders,
    rank_in_region
FROM
    RegionalProductProfit
WHERE
    rank_in_region <= 5 -- Select the bottom 5 products in each region
ORDER BY
    region,
    rank_in_region
"""

# Run the query
bottom_products_region_job = client.query(bottom_products_region_query)

# Convert results to DataFrame
bottom_products_region_df = bottom_products_region_job.to_dataframe()

print("\n--- Top 5 Worst Performing Products by Region (Based on Total Profit) ---")
display(bottom_products_region_df)


--- Top 5 Worst Performing Products by Region (Based on Total Profit) ---


,region,category,subcategory,product_name,total_profit,total_sales,total_quantity,number_of_orders,rank_in_region
0,Central,Office Supplies,Binders,GBC DocuBind P400 Electric Binding System,-3048.6176,8710.336,16,3,1
1,Central,Office Supplies,Binders,Fellowes PB500 Electric Punch Plastic Comb Bin...,-1525.1880,6100.752,12,3,2
2,Central,Office Supplies,Appliances,3.6 Cubic Foot Counter Height Office Refrigerator,-1378.8216,530.316,9,3,3
3,Central,Technology,Machines,Lexmark MX611dhe Monochrome Laser Printer,-1189.9930,14279.916,13,3,4
4,Central,Office Supplies,Binders,Ibico Hi-Tech Manual Binding System,-1189.4610,1829.940,18,5,5
5,East,Technology,Machines,Cubify CubeX 3D Printer Double Head Print,-9239.9692,6299.979,7,2,1
6,East,Office Supplies,Supplies,Martin Yale Chadless Opener Electric Letter Op...,-1199.2464,5329.984,8,2,2
7,East,Furniture,Tables,"Riverside Furniture Oval Coffee Table, Oval En...",-1187.5590,3958.530,23,3,3
8,East,Office Supplies,Binders,GBC Ibimaster 500 Manual ProClick Binding System,-1065.3720,5402.958,17,5,4
9,East,Technology,Machines,Cisco 9971 IP Video Phone Charcoal,-950.4000,1188.000,9,1,5


# V - Validation

## Check for completeness

In [ ]:
# prompt: convert this to python:
# SELECT
#   COUNT(*) as total_records,
#   COUNT(DISTINCT date) as unique_dates,
#   MIN(date) as earliest_date,
#   MAX(date) as latest_date
# FROM dataset

# Note: Assuming 'date' is the name of your date column and 'dataset' is the name of your table
# You may need to replace 'dataset' with the actual table ID if it's different.

query = f"""
SELECT
  COUNT(*) as total_records,
  COUNT(DISTINCT order_date) as unique_dates, -- Assuming 'order_date' is your date column
  MIN(order_date) as earliest_date,
  MAX(order_date) as latest_date
FROM
  `{FULL_TABLE_ID}`  -- Using the predefined FULL_TABLE_ID
"""

# Run the query
query_job = client.query(query)

# Convert results to DataFrame
results_df = query_job.to_dataframe()

# Display the results
print("--- Data Completeness Check ---")
display(results_df)

--- Data Completeness Check ---


,total_records,unique_dates,earliest_date,latest_date
0,9994,1236,2019-01-03,2022-12-30


## Check schema

In [ ]:
# Check the table schema to confirm column names
table = client.get_table(FULL_TABLE_ID)
print("--- Table Schema ---")
for field in table.schema:
    print(f"- {field.name} ({field.field_type})")

--- Table Schema ---
- order_id (STRING)
- order_date (DATE)
- ship_date (DATE)
- customer (STRING)
- manufactory (STRING)
- product_name (STRING)
- segment (STRING)
- category (STRING)
- subcategory (STRING)
- region (STRING)
- zip (INTEGER)
- city (STRING)
- state (STRING)
- country (STRING)
- discount (FLOAT)
- profit (FLOAT)
- quantity (INTEGER)
- sales (FLOAT)
- profit_margin (FLOAT)
- customer_id (INTEGER)


## Check for null values

In [ ]:
# Comprehensive null analysis
null_info = pd.DataFrame({
    'column_name': null_counts_df.columns,
    'null_count': null_counts_df.iloc[0], # Use the first row of null_counts_df for counts
    'null_percentage': (null_counts_df.iloc[0] / 24699) * 100, # Assuming 24699 is the total number of records from the data completeness check
    'total_rows': 24699 # Assuming 24699 is the total number of records
})

# Filter to show only columns with nulls
null_info = null_info[null_info['null_count'] > 0]
print(null_info)

Empty DataFrame
Columns: [column_name, null_count, null_percentage, total_rows]
Index: []


In [ ]:
# prompt: Perform null value analysis and calculate percentages

# SQL query to count null values for each column
query = f"""
SELECT
    COUNTIF(order_id IS NULL) AS null_order_id,
    COUNTIF(order_date IS NULL) AS null_order_date,
    COUNTIF(ship_date IS NULL) AS null_ship_date,
    COUNTIF(customer_id IS NULL) AS null_customer_id,
    COUNTIF(segment IS NULL) AS null_segment,
    COUNTIF(country IS NULL) AS null_country,
    COUNTIF(city IS NULL) AS null_city,
    COUNTIF(state IS NULL) AS null_state,
    COUNTIF(region IS NULL) AS null_region,
    COUNTIF(category IS NULL) AS null_category,
    COUNTIF(subcategory IS NULL) AS null_subcategory,
    COUNTIF(product_name IS NULL) AS null_product_name,
    COUNTIF(sales IS NULL) AS null_sales,
    COUNTIF(quantity IS NULL) AS null_quantity,
    COUNTIF(discount IS NULL) AS null_discount,
    COUNTIF(profit IS NULL) AS null_profit
FROM
  `{FULL_TABLE_ID}`
"""

# Run the query
query_job = client.query(query)

# Convert results to DataFrame
null_counts_df = query_job.to_dataframe()

# Get the total number of records (assuming we have this from a previous check, if not, you can run a COUNT(*) query)
# For now, using the number from cell 99ed91c4 output
total_records = 9994 # Replace with actual total if needed

# Comprehensive null analysis
null_info = pd.DataFrame({
    'column_name': null_counts_df.columns,
    'null_count': null_counts_df.iloc[0].values, # Use .values to get the counts as a numpy array
    'null_percentage': (null_counts_df.iloc[0].values / total_records) * 100,
    'total_rows': total_records
})

# Filter to show only columns with nulls
null_info = null_info[null_info['null_count'] > 0]

print("--- Null Value Analysis ---")
display(null_info)

--- Null Value Analysis ---


,column_name,null_count,null_percentage,total_rows


## Check for duplicate rows

In [ ]:
# prompt: check for duplicate rows

query = f"""
SELECT
    order_id,
    order_date,
    ship_date,
    customer_id,
    segment,
    country,
    city,
    state,
    region,
    category,
    subcategory,
    product_name,
    sales,
    quantity,
    discount,
    profit,
    COUNT(*) as duplicate_count
FROM
    `{FULL_TABLE_ID}`
GROUP BY
    order_id,
    order_date,
    ship_date,
    customer_id,
    segment,
    country,
    city,
    state,
    region,
    category,
    subcategory,
    product_name,
    sales,
    quantity,
    discount,
    profit
HAVING
    COUNT(*) > 1
"""

# Run the query
query_job = client.query(query)

# Convert results to DataFrame
duplicate_rows_df = query_job.to_dataframe()

# Display the results
print("--- Duplicate Row Check ---")
if duplicate_rows_df.empty:
    print("No duplicate rows found.")
else:
    display(duplicate_rows_df)

--- Duplicate Row Check ---


,order_id,order_date,ship_date,customer_id,segment,country,city,state,region,category,subcategory,product_name,sales,quantity,discount,profit,duplicate_count
0,US-2020-150119,2019-04-23,2019-04-27,452,Home Office,United States,Columbus,Ohio,East,Furniture,Chairs,Global Leather Highback Executive Chair with P...,281.372,2,0.3,-12.0588,2


## Check for outliers

In [ ]:
# prompt: check for outliers

# SQL query to select relevant numerical columns
query = f"""
SELECT
    sales,
    quantity,
    profit
FROM
    `{FULL_TABLE_ID}`
"""

# Run the query
query_job = client.query(query)

# Convert results to DataFrame
numerical_data_df = query_job.to_dataframe()

print("--- Descriptive Statistics for Numerical Columns ---")
display(numerical_data_df.describe())

# Identify outliers using IQR for 'sales', 'quantity', and 'profit'
numerical_cols = ['sales', 'quantity', 'profit']
outlier_info = {}

for col in numerical_cols:
    if col in numerical_data_df.columns:
        Q1 = numerical_data_df[col].quantile(0.25)
        Q3 = numerical_data_df[col].quantile(0.75)
        IQR = Q3 - Q1

        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        outliers = numerical_data_df[(numerical_data_df[col] < lower_bound) | (numerical_data_df[col] > upper_bound)]

        outlier_info[col] = {
            'lower_bound (IQR)': lower_bound,
            'upper_bound (IQR)': upper_bound,
            'number_of_outliers': len(outliers),
            'percentage_of_outliers': (len(outliers) / len(numerical_data_df)) * 100 if len(numerical_data_df) > 0 else 0,
            'outlier_examples': outliers.head() # Display first 5 outlier examples
        }

print("\n--- Outlier Analysis (using IQR) ---")
for col, info in outlier_info.items():
    print(f"\n--- '{col}' Outliers ---")
    print(f"  IQR Lower Bound: {info['lower_bound (IQR)']:.2f}")
    print(f"  IQR Upper Bound: {info['upper_bound (IQR)']:.2f}")
    print(f"  Number of Outliers: {info['number_of_outliers']}")
    print(f"  Percentage of Outliers: {info['percentage_of_outliers']:.2f}%")
    print("  Outlier Examples:")
    display(info['outlier_examples'])

--- Descriptive Statistics for Numerical Columns ---


,sales,quantity,profit
count,9994.000000,9994.0,9994.000000
mean,229.858001,3.789574,28.656896
std,623.245101,2.22511,234.260108
min,0.444000,1.0,-6599.978000
25%,17.280000,2.0,1.728750
50%,54.490000,3.0,8.666500
75%,209.940000,5.0,29.364000
max,22638.480000,14.0,8399.976000



--- Outlier Analysis (using IQR) ---

--- 'sales' Outliers ---
  IQR Lower Bound: -271.71
  IQR Upper Bound: 498.93
  Number of Outliers: 1167
  Percentage of Outliers: 11.68%
  Outlier Examples:


,sales,quantity,profit
20,872.320,4,244.2496
41,1199.980,2,467.9922
43,526.582,2,-52.6582
48,1325.850,5,238.6530
49,785.880,6,212.1876



--- 'quantity' Outliers ---
  IQR Lower Bound: -2.50
  IQR Upper Bound: 9.50
  Number of Outliers: 170
  Percentage of Outliers: 1.70%
  Outlier Examples:


,sales,quantity,profit
24,209.50,10,58.6600
104,189.70,10,91.0560
127,258.90,10,93.2040
221,259.74,13,124.6752
230,1564.29,13,406.7154



--- 'profit' Outliers ---
  IQR Lower Bound: -39.72
  IQR Upper Bound: 70.82
  Number of Outliers: 1881
  Percentage of Outliers: 18.82%
  Outlier Examples:


,sales,quantity,profit
9,354.90,5,88.7250
16,447.86,7,219.4514
20,872.32,4,244.2496
26,447.86,7,210.4942
38,387.99,1,182.3553


## Investigate outliers in 'profit'

In [ ]:
# prompt: investigate outliers in the 'profit' column

# Assuming outlier_info from the previous step (cell c71dcb22) is available
if 'outlier_info' in locals() and 'profit' in outlier_info:
    profit_outlier_bounds = outlier_info['profit']
    lower_bound = profit_outlier_bounds['lower_bound (IQR)']
    upper_bound = profit_outlier_bounds['upper_bound (IQR)']

    # SQL query to select rows where profit is outside the IQR bounds
    query = f"""
    SELECT
        *
    FROM
        `{FULL_TABLE_ID}`
    WHERE
        profit < {lower_bound} OR profit > {upper_bound}
    """

    # Run the query
    query_job = client.query(query)

    # Convert results to DataFrame
    profit_outliers_df = query_job.to_dataframe()

    print(f"--- Details of Outliers in 'profit' (outside {lower_bound:.2f} and {upper_bound:.2f}) ---")
    display(profit_outliers_df)

else:
    print("Profit outlier information not found. Please run the outlier analysis cell first.")

--- Details of Outliers in 'profit' (outside -39.72 and 70.82) ---


,order_id,order_date,ship_date,customer,manufactory,product_name,segment,category,subcategory,region,zip,city,state,country,discount,profit,quantity,sales,profit_margin,customer_id
0,US-2022-153269,2021-03-09,2021-03-12,Pamela Stobb,Other,"Situations Contoured Folding Chairs, 4/Set",Consumer,Furniture,Chairs,East,1810,Andover,Massachusetts,United States,0.0,88.7250,5,354.90,0.250,583
1,US-2020-152443,2019-05-21,2019-05-26,Frank Gastineau,GBC,GBC ProClick Punch Binding System,Home Office,Office Supplies,Binders,East,1841,Lawrence,Massachusetts,United States,0.0,219.4514,7,447.86,0.490,283
2,US-2020-166086,2019-05-07,2019-05-12,Carol Triggs,Other,"Lifetime Advantage Folding Chairs, 4/Carton",Consumer,Furniture,Chairs,East,1841,Lawrence,Massachusetts,United States,0.0,244.2496,4,872.32,0.280,132
3,US-2021-129322,2020-08-08,2020-08-10,Denny Blanton,GBC,GBC DocuBind P50 Personal Binding Machine,Consumer,Office Supplies,Binders,East,1841,Lawrence,Massachusetts,United States,0.0,210.4942,7,447.86,0.470,226
4,US-2023-132339,2022-08-19,2022-08-23,Jennifer Braxton,Fellowes,Fellowes PB300 Plastic Comb Binding Machine,Corporate,Office Supplies,Binders,East,1841,Lawrence,Massachusetts,United States,0.0,182.3553,1,387.99,0.470,359
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1876,US-2023-123967,2022-11-01,2022-11-03,Sarah Foster,Hon,Hon 94000 Series Round Tables,Consumer,Furniture,Tables,West,98226,Bellingham,Washington,United States,0.0,239.9058,9,2665.62,0.090,679
1877,US-2023-150420,2022-06-03,2022-06-08,Giulietta Dortch,Plantronics,Plantronics Audio 995 Wireless Stereo Headset,Corporate,Technology,Accessories,West,98226,Bellingham,Washington,United States,0.0,362.8350,10,1099.50,0.330,305
1878,US-2020-107818,2019-09-08,2019-09-14,Marc Crier,Hunt,"Hunt PowerHouse Electric Pencil Sharpener, Blue",Consumer,Office Supplies,Art,West,99301,Pasco,Washington,United States,0.0,91.1520,8,303.84,0.300,484
1879,US-2020-107818,2019-09-08,2019-09-14,Marc Crier,Hon,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",Consumer,Furniture,Chairs,West,99301,Pasco,Washington,United States,0.2,121.9900,5,975.92,0.125,484


In [226]:
# prompt: create a line chart that tracks profit outliers by region over time. use the previously generated outlier results.

import pandas as pd
# Ensure the 'profit_outliers_df' DataFrame exists from the previous steps
if 'profit_outliers_df' in locals() and not profit_outliers_df.empty:
    # Ensure the 'order_date' column is in datetime format
    profit_outliers_df['order_date'] = pd.to_datetime(profit_outliers_df['order_date'])

    # Sort by date for a meaningful time series plot
    profit_outliers_df = profit_outliers_df.sort_values(by='order_date')

    # Create a line chart for profit outliers over time, segmented by region
    fig = px.line(profit_outliers_df,
                  x='order_date',
                  y='profit',
                  color='region',  # Color lines by region
                  title='Profit Outliers by Region Over Time',
                  labels={'order_date': 'Date', 'profit': 'Profit ($)'})

    # Update layout for better readability
    fig.update_layout(
        xaxis_title='Date',
        yaxis_title='Profit ($)',
        plot_bgcolor='white'
    )

    # Show the plot
    fig.show()

else:
    print("The 'profit_outliers_df' DataFrame is not available or is empty. Please run the preceding steps to identify and load profit outliers.")


In [227]:
# prompt: analyze the data to identify whether there is a pattern with certain customers and outlier prices

import pandas as pd
# Ensure the 'profit_outliers_df' DataFrame exists from the previous steps
if 'profit_outliers_df' in locals() and not profit_outliers_df.empty:

    # Analysis 1: Identify customers with multiple outlier transactions
    customer_outlier_counts = profit_outliers_df['customer_id'].value_counts().reset_index()
    customer_outlier_counts.columns = ['customer_id', 'outlier_transaction_count']

    # Filter for customers with more than one outlier transaction
    frequent_outlier_customers = customer_outlier_counts[customer_outlier_counts['outlier_transaction_count'] > 1]

    print("\n--- Customers with Multiple Outlier Transactions in Profit ---")
    if not frequent_outlier_customers.empty:
        display(frequent_outlier_customers)
    else:
        print("No customers found with multiple outlier transactions in profit.")

    # Analysis 2: Investigate common attributes of outlier transactions
    # Group outliers by customer_id and analyze their characteristics
    customer_outlier_details = profit_outliers_df.groupby('customer_id').agg(
        total_outlier_profit=('profit', 'sum'),
        average_outlier_profit=('profit', 'mean'),
        min_outlier_profit=('profit', 'min'),
        max_outlier_profit=('profit', 'max'),
        total_sales_in_outliers=('sales', 'sum'),
        average_discount_in_outliers=('discount', 'mean'),
        number_of_outlier_transactions=('order_id', 'count'),
        unique_products_in_outliers=('product_name', lambda x: x.nunique()),
        unique_categories_in_outliers=('category', lambda x: x.nunique()),
        unique_regions_in_outliers=('region', lambda x: x.nunique())
    ).reset_index()

    # Merge with the frequent outlier customers if needed
    customer_outlier_summary = pd.merge(frequent_outlier_customers, customer_outlier_details, on='customer_id', how='left')


    print("\n--- Summary of Outlier Transactions per Customer ---")

    # Optional: Format monetary and percentage columns for better readability
    monetary_cols_customer_outliers = ['total_outlier_profit', 'average_outlier_profit', 'min_outlier_profit', 'max_outlier_profit', 'total_sales_in_outliers']
    for col in customer_outlier_summary.columns:
        if col in monetary_cols_customer_outliers:
            if pd.api.types.is_numeric_dtype(customer_outlier_summary[col]):
                customer_outlier_summary[col] = customer_outlier_summary[col].apply(
                    lambda x: f"-${abs(x):,.2f}" if pd.notnull(x) and x < 0
                    else f"${x:,.2f}" if pd.notnull(x) else None
                )

    percentage_cols_customer_outliers = ['average_discount_in_outliers']
    for col in customer_outlier_summary.columns:
        if col in percentage_cols_customer_outliers:
             if pd.api.types.is_numeric_dtype(customer_outlier_summary[col]):
                customer_outlier_summary[col] = customer_outlier_summary[col].apply(lambda x: f"{x:.2%}" if pd.notnull(x) else None)


    display(customer_outlier_summary)

    # Analysis 3: Examine the products, categories, subcategories, and regions involved in outlier transactions
    print("\n--- Top Products in Outlier Transactions (by Occurrence) ---")
    display(profit_outliers_df['product_name'].value_counts().head(10).reset_index(name='outlier_count'))

    print("\n--- Top Categories in Outlier Transactions (by Occurrence) ---")
    display(profit_outliers_df['category'].value_counts().head(10).reset_index(name='outlier_count'))

    print("\n--- Top Subcategories in Outlier Transactions (by Occurrence) ---")
    display(profit_outliers_df['subcategory'].value_counts().head(10).reset_index(name='outlier_count'))

    print("\n--- Top Regions in Outlier Transactions (by Occurrence) ---")
    display(profit_outliers_df['region'].value_counts().head(10).reset_index(name='outlier_count'))

    # Analysis 4: Correlation between discount and outlier profit
    print("\n--- Correlation between Discount and Profit in Outlier Transactions ---")
    # Ensure columns are numeric before calculating correlation
    outlier_correlation = profit_outliers_df[['discount', 'profit']].corr()
    display(outlier_correlation)

    # You could also visualize the relationship between discount and profit for outliers
    fig = px.scatter(profit_outliers_df,
                     x='discount',
                     y='profit',
                     color='region', # Color points by region
                     hover_data=['customer_id', 'product_name', 'category', 'subcategory'], # Add hover information
                     title='Discount vs. Profit for Outlier Transactions',
                     labels={'discount': 'Discount', 'profit': 'Profit ($)'})

    fig.update_layout(plot_bgcolor='white')
    fig.show()


else:
    print("The 'profit_outliers_df' DataFrame is not available or is empty. Please run the preceding steps to identify and load profit outliers.")



--- Customers with Multiple Outlier Transactions in Profit ---


,customer_id,outlier_transaction_count
0,110,9
1,635,9
2,690,9
3,442,9
4,170,9
...,...,...
500,560,2
501,241,2
502,41,2
503,742,2



--- Summary of Outlier Transactions per Customer ---


,customer_id,outlier_transaction_count,total_outlier_profit,average_outlier_profit,min_outlier_profit,max_outlier_profit,total_sales_in_outliers,average_discount_in_outliers,number_of_outlier_transactions,unique_products_in_outliers,unique_categories_in_outliers,unique_regions_in_outliers
0,110,9,"$1,952.42",$216.94,$81.43,$473.61,"$6,060.25",0.00%,9,8,2,2
1,635,9,"$1,324.88",$147.21,-$86.37,$654.76,"$6,043.23",20.00%,9,9,2,2
2,690,9,$960.04,$106.67,-$131.95,$609.72,"$8,545.63",22.22%,9,9,2,2
3,442,9,$504.23,$56.03,-$356.73,$327.51,"$12,558.11",24.44%,9,9,3,3
4,170,9,"$1,715.97",$190.66,$78.67,$523.71,"$8,675.35",8.89%,9,9,3,3
...,...,...,...,...,...,...,...,...,...,...,...,...
500,560,2,"-$2,334.18","-$1,167.09","-$2,287.78",-$46.40,"$1,716.25",55.00%,2,2,2,1
501,241,2,$650.94,$325.47,$139.57,$511.37,"$2,846.50",5.00%,2,2,2,1
502,41,2,$120.41,$60.20,-$93.33,$213.74,"$1,536.94",25.00%,2,2,1,1
503,742,2,$246.57,$123.29,$102.94,$143.63,$908.90,0.00%,2,2,2,1



--- Top Products in Outlier Transactions (by Occurrence) ---


...

,subcategory,outlier_count
0,Phones,264
1,Chairs,225
2,Binders,209
3,Tables,208
4,Accessories,160
5,Storage,156
6,Appliances,137
7,Paper,122
8,Furnishings,100
9,Bookcases,94



--- Top Regions in Outlier Transactions (by Occurrence) ---


,region,outlier_count
0,East,600
1,West,495
2,Central,481
3,South,305



--- Correlation between Discount and Profit in Outlier Transactions ---


,discount,profit
discount,1.000000,-0.413361
profit,-0.413361,1.000000
